In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import time
from typing import Any, List, Dict, Set
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# Clear old model variables and free GPU memory before loading a model. This might be more of a precaution than a requirement.
def pre_run_cleanup():
    for name in [
        "model",
        "tokenizer",
        "model_inputs",
        "generated_ids",
        "raw_output",
        "generated_sql",
        "response",
        "prompt",
        "messages",
    ]:
        if name in globals():
            del globals()[name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [ ]:
pre_run_cleanup()

In [ ]:
# Paths and folders
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

for folder in [DATA_DIR, RESULTS_DIR]:
    folder.mkdir(exist_ok=True)

# Model config
MODEL_CONFIGS = {
    "3b_general": {"model_name": "Qwen/Qwen2.5-3B-Instruct", "model_role": "general", "model_size_b": 3},
    "3b_code": {"model_name": "Qwen/Qwen2.5-Coder-3B-Instruct", "model_role": "code", "model_size_b": 3},
    "7b_general": {"model_name": "Qwen/Qwen2.5-7B-Instruct", "model_role": "general", "model_size_b": 7},
    "7b_code": {"model_name": "Qwen/Qwen2.5-Coder-7B-Instruct", "model_role": "code", "model_size_b": 7},
}
 
SELECTED_MODEL_KEY = "3b_general" # Model used in run
REPEAT_ID = "r10"                 # Repeated run identifier
TOP_K = 3                         # Number of retrieved schema chunks

SELECTED_MODEL = MODEL_CONFIGS[SELECTED_MODEL_KEY]

MODEL_NAME = SELECTED_MODEL["model_name"]
MODEL_ROLE = SELECTED_MODEL["model_role"]
MODEL_SIZE_B = SELECTED_MODEL["model_size_b"]

RUN_ID = f"{SELECTED_MODEL_KEY}_{REPEAT_ID}"

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if MODEL_SIZE_B == 3:
    # 3B models were loaded without quantization in the experiment.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

elif MODEL_SIZE_B == 7:
    # 7B models were loaded with 4-bit NF4 quantization to fit the available GPU memory.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

In [ ]:
# Load input data
QUESTION_FILE = DATA_DIR / "New_ALARM_ORDER_questions.csv"
SCHEMA_CHUNKS_FILE = DATA_DIR / "schema_chunks_alarm_order_v1.json"

question_bank = pd.read_csv(QUESTION_FILE, sep=";", encoding="utf-8-sig")

with open(SCHEMA_CHUNKS_FILE, "r", encoding="utf-8") as f:
    schema_chunks = json.load(f)

print(f"Loaded {len(question_bank)} questions")
print(f"Loaded {len(schema_chunks)} schema chunks")

In [ ]:
def tokenize(text: str) -> Set[str]:
    return set(re.findall(r"\w+", str(text).lower()))


def build_text_for_scoring(chunk: Dict[str, Any]) -> str:
    chunk_id = chunk.get("chunk_id", "")
    table = chunk.get("table", "")
    text = chunk.get("text", "")
    return f"{chunk_id} {table} {text}"


def detect_domain(question_words: Set[str]) -> str:
    # Example of generic domain keyword list
    alarm_terms = {
        "alarm", "alarms", "equipment", "machine", "active",
        "start", "end", "duration", "recurring"
    }
    
    order_terms = {
        "order", "orders", "item", "status", "planned",
        "actual", "quantity", "difference", "incomplete"
    }

    alarm_score = len(question_words & alarm_terms)
    order_score = len(question_words & order_terms)

    # Alarm is the default domain if no order-specific terms dominate.
    if order_score > alarm_score:
        return "order"
    return "alarm"


def get_always_include_ids(domain: str) -> Set[str]:
    if domain == "order":
        return {"orders_joins", "order_terms_en_sv"}
    return {"alarm_joins", "alarm_terms_en_sv"}


def score_chunk(question_words: Set[str], chunk: Dict[str, Any], always_ids: Set[str]) -> int:
    chunk_id = chunk.get("chunk_id", "")
    score_text = build_text_for_scoring(chunk)

    text_words = tokenize(score_text)
    chunk_id_words = tokenize(chunk_id.replace("_", " "))

    score = 0
    score += len(question_words & text_words)
    score += 2 * len(question_words & chunk_id_words)

    if chunk_id.endswith("_pattern"):
        score += 1
    elif chunk_id.endswith("_table"):
        score += 0
    elif chunk_id.endswith("_joins") or chunk_id.endswith("_en_sv"):
        score += 1

    if chunk_id in always_ids:
        score += 100

    return score


def retrieve_context(question_text: str, schema_chunks: List[Dict[str, Any]], top_k: int = 3) -> List[Dict[str, Any]]:
    question_words = tokenize(question_text)
    domain = detect_domain(question_words)
    always_include_ids = get_always_include_ids(domain)

    scored_chunks: list[tuple[int, dict[str, Any]]] = []
    for chunk in schema_chunks:
        score = score_chunk(question_words, chunk, always_include_ids)
        scored_chunks.append((score, chunk))

    scored_chunks.sort(key=lambda x: x[0], reverse=True)

    selected: list[dict[str, Any]] = []
    seen: set[str] = set()

    for score, chunk in scored_chunks:
        chunk_id = chunk["chunk_id"]

        if chunk_id in always_include_ids or score > 0:
            if chunk_id not in seen:
                selected.append(chunk)
                seen.add(chunk_id)

        non_always_count = sum(1 for c in selected if c["chunk_id"] not in always_include_ids)
        if non_always_count >= top_k and always_include_ids.issubset(seen):
            break

    return selected

# Complete prompt template given to all models and runs.
def build_prompt(question_text: str, retrieved_chunks: list[dict[str, Any]]) -> str:
    context_text = "\n\n".join(chunk["text"] for chunk in retrieved_chunks)

    return f"""
You are generating SQL for Microsoft SQL Server.

The user question may be written in Swedish.
Translate it into exactly one SQL query.

Rules:
- Return ONLY SQL.
- Do NOT explain anything.
- Do NOT include markdown.
- Use ONLY SQL Server syntax.
- Use ONLY tables, columns, joins, and schema names supported by the schema context.
- Do NOT invent tables, columns, or placeholder names.
- Follow the user question as literally as possible.
- Do NOT add extra columns unless explicitly requested.
- Do NOT add row limits unless explicitly requested.
- If a row limit is explicitly requested, use TOP, never LIMIT.
- Use the verified join path exactly as given in the schema context.
- If the user asks for “most common” or similar frequency-based results, return grouped results without TOP unless a limit is explicitly requested.
- If the schema context is insufficient, return exactly INVALID_QUERY.
- Do NOT rename columns.


Schema context:
{context_text}

User question:
{question_text}


SQL:
""".strip()

def extract_sql(raw_output: str) -> str:
    if not isinstance(raw_output, str):
        return ""
    
    text = raw_output.strip()
    
    if text == "INVALID_QUERY":
        return "INVALID_QUERY"
    
    match = re.search(r"```sql\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    
    match = re.search(r"```\s*(.*?)```", text, flags=re.DOTALL)
    if match:
        inner = match.group(1).strip()
        if re.match(r"(?is)^(SELECT|WITH)\b", inner):
            return inner
    
    match = re.search(r"(?is)\b(SELECT|WITH)\b.*", text)
    if match:
        sql = match.group(0).strip()
        sql = sql.split("```")[0].strip()
        return sql
    
    return ""

In [ ]:
# Generates a model response and records token counts.
def hf_generate_sql(prompt: str) -> tuple[str, int, int]:
    messages = [
        {
            "role": "system",
            "content": "You generate exactly one SQL Server SELECT query and nothing else."
        },
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=256,
            do_sample=False
        )

    new_tokens = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    completion_tokens = len(new_tokens)

    return response, prompt_tokens, completion_tokens

In [ ]:
def run_generation_pipeline(
    question_bank: pd.DataFrame,
    schema_chunks: list[dict[str, Any]],
    generate_sql_fn,
    model_name: str,
    model_role: str,
    model_size_b: int,
    run_id: str,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    
    for _, row in question_bank.iterrows():
        question_id = row["id"]
        question_text = row["question"]

        start_time = time.perf_counter()

        raw_output = ""
        generated_sql = ""
        retrieved_chunks: list[dict[str, Any]] = []
        # Continue the run even if one question fails.
        try:
            retrieved_chunks = retrieve_context(question_text, schema_chunks, top_k=top_k)
            prompt = build_prompt(question_text, retrieved_chunks)
            
            raw_output, prompt_tokens, completion_tokens = generate_sql_fn(prompt)
            generated_sql = extract_sql(raw_output)

            # SQL validation is performed in the execution notebook.
            validation_note = "not_validated_in_generation"

            latency = time.perf_counter() - start_time
            total_tokens = prompt_tokens + completion_tokens

            rows.append({
                "run_id": run_id,
                "model_name": model_name,
                "model_role": model_role,
                "model_size_b": model_size_b,
                "question_id": question_id,
                "question_text": question_text,
                "retrieved_chunk_ids": "|".join(chunk["chunk_id"] for chunk in retrieved_chunks),
                "raw_model_output": raw_output,
                "generated_sql": generated_sql,
                "sql_validation_note": validation_note,
                "generation_latency_seconds": latency,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens,
                "error_message": "",
            })

        except Exception as e:
            latency = time.perf_counter() - start_time

            rows.append({
                "run_id": run_id,
                "model_name": model_name,
                "model_role": model_role,
                "model_size_b": model_size_b,
                "question_id": question_id,
                "question_text": question_text,
                "retrieved_chunk_ids": "|".join(chunk["chunk_id"] for chunk in retrieved_chunks) if retrieved_chunks else "",
                "raw_model_output": raw_output,
                "generated_sql": "",
                "sql_validation_note": "exception",
                "generation_latency_seconds": latency,
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0,
                "error_message": str(e),
            })

    return pd.DataFrame(rows)

In [ ]:
generation_results_df = run_generation_pipeline(
    question_bank=question_bank,
    schema_chunks=schema_chunks,
    generate_sql_fn=hf_generate_sql,
    model_name=MODEL_NAME,
    model_role=MODEL_ROLE,
    model_size_b=MODEL_SIZE_B,
    run_id=RUN_ID,
    top_k=TOP_K,
)

In [ ]:
output_csv = RESULTS_DIR / f"{RUN_ID}_generation.csv"
generation_results_df.to_csv(output_csv, index=False, encoding="utf-8")

print(f"Saved CSV: {output_csv}")